In [1]:
# import libraries
import requests
import pandas as pd
import numpy as np
from time import sleep
import json
# from ete4 import NCBITaxa

In [3]:
# read in data
ncbi = pd.read_csv('/Users/bellamia/Projects/passion_projects/pathology_genomics/spillover_model/data/raw/sequences.csv')

# Quick overview
print(ncbi.shape)
print(ncbi.columns.tolist())
ncbi.head()

/var/folders/sd/p80bpffj1hlgflffbd1p2r_w0000gn/T/ipykernel_27971/3703297220.py:2: DtypeWarning: Columns (3,4,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  ncbi = pd.read_csv('/Users/bellamia/Projects/passion_projects/pathology_genomics/spillover_model/data/raw/sequences.csv')


(3256603, 14)
['Accession', 'Organism_Name', 'Species', 'Genotype', 'Isolate', 'Length', 'Nuc_Completeness', 'Geo_Location', 'Host', 'Tissue_Specimen_Source', 'Submitters', 'Collection_Date', 'Release_Date', 'Molecule_type']


,Accession,Organism_Name,Species,Genotype,Isolate,Length,Nuc_Completeness,Geo_Location,Host,Tissue_Specimen_Source,Submitters,Collection_Date,Release_Date,Molecule_type
0,NC_138438,Red panda feces-associated circular DNA virus 12,Alohovirus ailgensis,NaN,Rpf011unssDNA01-5,2820,complete,China: Sichuan Province,Ailurus fulgens,feces,"Zhao,M., Yue,C., Yang,Z., Li,Y., Zhang,D., Zha...",2020-05,2026-04-07,ssDNA
1,NC_138461,Red panda feces-associated circular DNA virus 11,Protegovirus ailuris,NaN,AliP02cress04-2015,2497,complete,China: Sichuan Province,Ailurus fulgens,feces,"Zhao,M., Yue,C., Yang,Z., Li,Y., Zhang,D., Zha...",2015,2026-04-07,ssDNA
2,NC_097195,Pacific flying fox faeces associated circular ...,Pekavirus fuais,NaN,Tbat_38855,2214,complete,Tonga,Pteropus tonganus,feces,"Male,M.F., Kraberger,S., Stainton,D., Kami,V.,...",2014,2026-04-01,ssDNA
3,NC_099061,Bat circovirus POA/2012/V,Vegetinivirus molmolis,NaN,cluster V,1728,complete,Brazil,Chiroptera,feces,"Lima,F.E., Cibulski,S.P., Dos Santos,H.F., Tei...",2012-02-13,2026-04-01,ssDNA
4,NC_116632,Delphin virus 1,Baranavirus uduis,NaN,3_2016_1939,1939,complete,Saint Vincent and the Grenadines,Orcinus orca,muscle,"Smith,K., Fielding,R., Schiavone,K., Hall,K., ...",2015-08-26,2026-04-01,ssDNA


In [5]:
class TimeTreeExtractor:
    def __init__(self):
        self.base_url = "http://timetree.org/api"
        self.session = requests.Session()

    def get_divergence_time(self, species1, species2):
        """
        Get divergence time between two species

        Args:
            species1 (str): Scientific name of first species
            species2 (str): Scientific name of second species

        Returns:
            float: Divergence time in millions of years ago (MYA)
        """
        sp1_clean = species1.replace(" ", "%20")
        sp2_clean = species2.replace(" ", "%20")
        url = f"{self.base_url}/pairwise/{sp1_clean}/{sp2_clean}"

        try:
            response = self.session.get(url, timeout=30)
            response.raise_for_status()
            data = response.json()

            if 'divergence_time' in data:
                result = float(data['divergence_time'])
            else:
                print(f"No divergence time found for {species1} vs {species2}")
                result = None

        except requests.exceptions.RequestException as e:
            print(f"API request failed: {e}")
            result = None
        except json.JSONDecodeError:
            print(f"Invalid JSON response for {species1} vs {species2}")
            result = None

        sleep(1)  # Rate limiting - be respectful to the API
        return result

    def _estimate_distance(self, species1, species2):
        """Fallback estimate when API fails."""
        return np.nan

    def calculate_phylogenetic_matrix(self, species_list):
        """
        Create phylogenetic distance matrix for list of species

        Args:
            species_list (list): List of scientific species names

        Returns:
            pandas.DataFrame: Symmetric matrix of phylogenetic distances
        """
        n_species = len(species_list)
        matrix = np.zeros((n_species, n_species))

        print(f"Calculating distances for {n_species} species...")

        for i, sp1 in enumerate(species_list):
            for j, sp2 in enumerate(species_list):
                if i == j:
                    matrix[i, j] = 0
                elif i < j:
                    divergence = self.get_divergence_time(sp1, sp2)
                    if divergence is not None:
                        matrix[i, j] = divergence
                        matrix[j, i] = divergence
                    else:
                        matrix[i, j] = self._estimate_distance(sp1, sp2)
                        matrix[j, i] = matrix[i, j]

                if (i * n_species + j + 1) % 10 == 0:
                    progress = ((i * n_species + j + 1) / (n_species ** 2)) * 100
                    print(f"Progress: {progress:.1f}%")

        df_matrix = pd.DataFrame(matrix, index=species_list, columns=species_list)
        return df_matrix

In [6]:
def calculate_human_distances(host_species_list):
    """
    Calculate phylogenetic distances from each host to humans

    Args:
        host_species_list (list): List of host species scientific names

    Returns:
        pandas.DataFrame: Host species with human distances and risk scores
    """
    extractor = TimeTreeExtractor()
    human_species = "Homo sapiens"
    results = []

    for host in host_species_list:
        print(f"Processing {host}...")
        divergence_mya = extractor.get_divergence_time(host, human_species)

        if divergence_mya is not None:
            phylo_risk_score = np.exp(-divergence_mya / 50)  # 50 MYA half-life

            if divergence_mya < 20:
                risk_category = "Very High"
            elif divergence_mya < 50:
                risk_category = "High"
            elif divergence_mya < 100:
                risk_category = "Medium"
            else:
                risk_category = "Low"

            results.append({
                'host_species': host,
                'human_divergence_mya': divergence_mya,
                'phylogenetic_risk_score': phylo_risk_score,
                'risk_category': risk_category
            })
        else:
            results.append({
                'host_species': host,
                'human_divergence_mya': None,
                'phylogenetic_risk_score': 0.1,
                'risk_category': 'Unknown'
            })

    return pd.DataFrame(results)


host_species = [
    "Rhinolophus sinicus",  # Chinese horseshoe bat
    "Manis javanica",       # Pangolin
    "Sus scrofa",           # Wild boar
    "Felis catus",          # Domestic cat
    "Rattus norvegicus",    # Norway rat
    "Eptesicus fuscus"      # Big brown bat
]

human_distance_df = calculate_human_distances(host_species)
print(human_distance_df)

Processing Rhinolophus sinicus...
API request failed: Expecting value: line 1 column 1 (char 0)
Processing Manis javanica...
API request failed: Expecting value: line 1 column 1 (char 0)
Processing Sus scrofa...
API request failed: Expecting value: line 1 column 1 (char 0)
Processing Felis catus...
API request failed: Expecting value: line 1 column 1 (char 0)
Processing Rattus norvegicus...
API request failed: Expecting value: line 1 column 1 (char 0)
Processing Eptesicus fuscus...
API request failed: Expecting value: line 1 column 1 (char 0)
          host_species human_divergence_mya  phylogenetic_risk_score  \
0  Rhinolophus sinicus                 None                      0.1   
1       Manis javanica                 None                      0.1   
2           Sus scrofa                 None                      0.1   
3          Felis catus                 None                      0.1   
4    Rattus norvegicus                 None                      0.1   
5     Eptesicus fus

In [7]:
# Get unique hosts from dataset
host_species_list = ncbi['Host'].dropna().unique().tolist()

# Calculate phylogenetic risk vs humans for all hosts
human_distance_df = calculate_human_distances(host_species_list)

# Merge back into main dataframe
ncbi = ncbi.merge(human_distance_df, left_on='Host', right_on='host_species', how='left').drop(columns='host_species')

print(ncbi[['Host', 'human_divergence_mya', 'phylogenetic_risk_score', 'risk_category']].drop_duplicates().sort_values('phylogenetic_risk_score', ascending=False).head(20))

Processing Ailurus fulgens...
API request failed: Expecting value: line 1 column 1 (char 0)
Processing Pteropus tonganus...
API request failed: Expecting value: line 1 column 1 (char 0)
Processing Chiroptera...
API request failed: Expecting value: line 1 column 1 (char 0)
Processing Orcinus orca...
API request failed: Expecting value: line 1 column 1 (char 0)
Processing Miniopterus fuliginosus...
API request failed: Expecting value: line 1 column 1 (char 0)
Processing Plecotus auritus...
API request failed: Expecting value: line 1 column 1 (char 0)
Processing Myotis emarginatus...
API request failed: Expecting value: line 1 column 1 (char 0)
Processing Marmota flaviventris...
API request failed: Expecting value: line 1 column 1 (char 0)
Processing Rhinolophus hipposideros...
API request failed: Expecting value: line 1 column 1 (char 0)
Processing Murina leucogaster...
API request failed: Expecting value: line 1 column 1 (char 0)
Processing Hydrochoerus hydrochaeris...
API request faile

KeyboardInterrupt: 